# Notebook 15 - SHAP Interpretability (No Scripts)

This notebook computes SHAP-based feature importance for a tree model across random split, GroupKFold, and spatial holdout protocols, and saves protocol-wise RMSE/R2 + SHAP artifacts.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import shap

NOTEBOOK_ROOT = Path.cwd() if Path.cwd().name == '15_shap_interpretability_no_scripts' else Path(r'c:/Users/obalo/Downloads/PhDOneDrive/PhD_Research_Operating_System/github_org_bootstrap/phd-geothermal-ml/manual_bootstrap/step_by_step_notebooks/15_shap_interpretability_no_scripts')
PROJECT_ROOT = NOTEBOOK_ROOT.parents[2]
PHD_OS_ROOT = PROJECT_ROOT.parents[1]
OUTPUT_ROOT = NOTEBOOK_ROOT / 'outputs' / 'summary'
TABLE_ROOT = OUTPUT_ROOT / 'tables'
FIG_ROOT = OUTPUT_ROOT / 'figures'
REPORT_ROOT = OUTPUT_ROOT / 'reports'
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

data_candidates = [
    PROJECT_ROOT / 'datasets' / 'geothermal_canonical_v2_0_0.csv',
    PHD_OS_ROOT / '03_data_processed' / 'geothermal_canonical_v2_0_0.csv',
    PHD_OS_ROOT / '02_data_raw' / 'geothermal_canonical_v2_0_0.csv',
    PHD_OS_ROOT / 'incoming_from_legacy' / 'geothermal_canonical_v2_0_0.csv',
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find geothermal_canonical_v2_0_0.csv in expected locations.')

df = pd.read_csv(data_path)
TARGET = 'Temperature(C)'
GROUP_COL = 'State'

# Exclude GROUP_COL from predictors to avoid leakage-like shortcut effects in interpretation
feature_cols = [c for c in df.columns if c not in [TARGET, GROUP_COL]]
X = df[feature_cols].copy()
y = df[TARGET].copy()
groups = df[GROUP_COL].astype(str).copy()

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

prep = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
], remainder='drop')

def to_dense(m):
    if sparse.issparse(m):
        return m.toarray()
    return np.asarray(m)

def run_protocol(protocol_name, tr_idx, te_idx, shap_sample=600):
    X_tr_raw = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    X_te_raw = X.iloc[te_idx]
    y_te = y.iloc[te_idx]

    X_tr = prep.fit_transform(X_tr_raw)
    X_te = prep.transform(X_te_raw)
    fn = prep.get_feature_names_out()

    X_tr_d = to_dense(X_tr)
    X_te_d = to_dense(X_te)

    reg = GradientBoostingRegressor(random_state=42, n_estimators=350, learning_rate=0.05, max_depth=3)
    reg.fit(X_tr_d, y_tr)
    pred = reg.predict(X_te_d)

    rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
    r2 = float(r2_score(y_te, pred))

    n_sample = min(shap_sample, X_te_d.shape[0])
    sample_idx = np.random.RandomState(42).choice(X_te_d.shape[0], size=n_sample, replace=False)
    X_shap = X_te_d[sample_idx]

    explainer = shap.TreeExplainer(reg)
    sv = explainer.shap_values(X_shap)
    mean_abs = np.abs(sv).mean(axis=0)

    imp = pd.DataFrame({
        'protocol': protocol_name,
        'feature': fn,
        'mean_abs_shap': mean_abs
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    perf = pd.DataFrame([{
        'protocol': protocol_name,
        'rmse': rmse,
        'r2': r2,
        'n_train': len(tr_idx),
        'n_test': len(te_idx),
        'shap_sample_size': n_sample
    }])

    return perf, imp

perf_rows = []
imp_rows = []

# random split (single representative seed for SHAP diagnostics)
idx = np.arange(len(X))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, random_state=0)
p, i = run_protocol('random_split', tr_idx, te_idx)
perf_rows.append(p)
imp_rows.append(i)

# GroupKFold (first fold)
gkf = GroupKFold(n_splits=5)
tr_idx, te_idx = next(iter(gkf.split(X, y, groups=groups)))
p, i = run_protocol('groupkfold', tr_idx, te_idx)
perf_rows.append(p)
imp_rows.append(i)

# Spatial holdout (first eligible state)
eligible = groups.value_counts()
eligible_states = eligible[eligible >= 30].index.tolist()
state = eligible_states[0]
te_mask = groups == state
tr_idx = np.where(~te_mask.values)[0]
te_idx = np.where(te_mask.values)[0]
p, i = run_protocol('spatial_holdout', tr_idx, te_idx)
p['holdout_state'] = state
perf_rows.append(p)
imp_rows.append(i)

perf_df = pd.concat(perf_rows, ignore_index=True)
imp_df = pd.concat(imp_rows, ignore_index=True)
top20 = imp_df.groupby('protocol', group_keys=False).head(20).reset_index(drop=True)

pivot = imp_df.pivot_table(index='feature', columns='protocol', values='mean_abs_shap', aggfunc='mean').fillna(0.0)
pivot['mean_all_protocols'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean_all_protocols', ascending=False)

perf_df.to_csv(TABLE_ROOT / 'protocol_performance_with_shap.csv', index=False)
imp_df.to_csv(TABLE_ROOT / 'all_feature_shap_importance_by_protocol.csv', index=False)
top20.to_csv(TABLE_ROOT / 'top20_feature_shap_by_protocol.csv', index=False)
pivot.to_csv(TABLE_ROOT / 'feature_shap_protocol_pivot.csv')

# Plot top 12 overall features
top_features = pivot.head(12).index.tolist()
plot_df = pivot.loc[top_features, ['random_split', 'groupkfold', 'spatial_holdout']].copy()
plot_df = plot_df.reset_index().rename(columns={'index': 'feature'})

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(plot_df))
w = 0.26
ax.bar(x - w, plot_df['random_split'], width=w, label='random_split')
ax.bar(x, plot_df['groupkfold'], width=w, label='groupkfold')
ax.bar(x + w, plot_df['spatial_holdout'], width=w, label='spatial_holdout')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['feature'], rotation=75, ha='right')
ax.set_ylabel('Mean |SHAP|')
ax.set_title('Top SHAP Features Across Protocols')
ax.legend()
fig.tight_layout()
fig_path = FIG_ROOT / 'shap_top_features_across_protocols.png'
fig.savefig(fig_path, dpi=600)
plt.close(fig)

lines = [
    '# Notebook 15 SHAP Interpretation Notes',
    '',
    '## Protocol Performance (RMSE and R2)',
]
for _, r in perf_df.iterrows():
    extra = f", holdout_state={r['holdout_state']}" if 'holdout_state' in r and pd.notna(r['holdout_state']) else ''
    lines.append(f"- {r['protocol']}: RMSE={r['rmse']:.4f}, R2={r['r2']:.4f}{extra}")

lines += ['', '## Top 10 Features By Mean SHAP Across Protocols']
for f in pivot.head(10).index:
    rv = pivot.loc[f, 'random_split'] if 'random_split' in pivot.columns else 0.0
    gv = pivot.loc[f, 'groupkfold'] if 'groupkfold' in pivot.columns else 0.0
    sv = pivot.loc[f, 'spatial_holdout'] if 'spatial_holdout' in pivot.columns else 0.0
    lines.append(f"- {f}: random={rv:.4f}, groupkfold={gv:.4f}, spatial={sv:.4f}")

(REPORT_ROOT / 'shap_interpretation_notes.md').write_text('\n'.join(lines), encoding='utf-8')

print('Saved:', TABLE_ROOT / 'protocol_performance_with_shap.csv')
print('Saved:', TABLE_ROOT / 'all_feature_shap_importance_by_protocol.csv')
print('Saved:', TABLE_ROOT / 'top20_feature_shap_by_protocol.csv')
print('Saved:', TABLE_ROOT / 'feature_shap_protocol_pivot.csv')
print('Saved:', fig_path)
print('Saved:', REPORT_ROOT / 'shap_interpretation_notes.md')
perf_df

c:\Users\obalo\Downloads\PhDOneDrive\PhD_Research_Operating_System\05_models\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved: c:\Users\obalo\Downloads\PhDOneDrive\PhD_Research_Operating_System\github_org_bootstrap\phd-geothermal-ml\manual_bootstrap\step_by_step_notebooks\15_shap_interpretability_no_scripts\outputs\summary\tables\protocol_performance_with_shap.csv
Saved: c:\Users\obalo\Downloads\PhDOneDrive\PhD_Research_Operating_System\github_org_bootstrap\phd-geothermal-ml\manual_bootstrap\step_by_step_notebooks\15_shap_interpretability_no_scripts\outputs\summary\tables\all_feature_shap_importance_by_protocol.csv
Saved: c:\Users\obalo\Downloads\PhDOneDrive\PhD_Research_Operating_System\github_org_bootstrap\phd-geothermal-ml\manual_bootstrap\step_by_step_notebooks\15_shap_interpretability_no_scripts\outputs\summary\tables\top20_feature_shap_by_protocol.csv
Saved: c:\Users\obalo\Downloads\PhDOneDrive\PhD_Research_Operating_System\github_org_bootstrap\phd-geothermal-ml\manual_bootstrap\step_by_step_notebooks\15_shap_interpretability_no_scripts\outputs\summary\tables\feature_shap_protocol_pivot.csv
Saved:

,protocol,rmse,r2,n_train,n_test,shap_sample_size,holdout_state
0,random_split,20.284542,0.761606,2536,634,600,NaN
1,groupkfold,31.418382,-0.081691,1933,1237,600,NaN
2,spatial_holdout,31.418382,-0.081691,1933,1237,600,nevada
